In [1]:
# CELL 1: Day 4 - Setup and Environment
# This cell installs required packages and sets up the environment for Day 4
# Completely independent from previous days

import os
import sys
import warnings
import subprocess
import importlib.util
from datetime import datetime

os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings("ignore")

print("DAY 4: AGENT ORCHESTRATION WITH LANGGRAPH")
print("=" * 60)
print(f"Started at: {datetime.now().isoformat()}")

required_packages = [
    ("langgraph", "langgraph"),
    ("langchain", "langchain"),
    ("langchain_core", "langchain-core"),
    ("langchain_community", "langchain-community"),
    ("wikipediaapi", "wikipedia-api"),
    ("requests", "requests"),
    ("pydantic", "pydantic"),
    ("typing_extensions", "typing_extensions"),
    ("gradio", "gradio"),
]

def install_package(package_name):
    try:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", package_name],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL
        )
        return True
    except subprocess.CalledProcessError:
        return False

def check_and_install_packages():
    installed = []
    failed = []
    
    for import_name, pip_name in required_packages:
        spec = importlib.util.find_spec(import_name)
        if spec is None:
            print(f"Installing: {pip_name}")
            if install_package(pip_name):
                installed.append(pip_name)
            else:
                failed.append(pip_name)
        else:
            print(f"Already installed: {pip_name}")
    
    if installed:
        print(f"\nSuccessfully installed: {', '.join(installed)}")
    if failed:
        print(f"\nFailed to install: {', '.join(failed)}")
    
    return len(failed) == 0

print("\nChecking and installing required packages...")
print("-" * 40)
installation_success = check_and_install_packages()
print("-" * 40)

if installation_success:
    print("All packages are ready.")
else:
    print("Some packages failed to install.")

print("\n" + "=" * 60)
print("Day 4 environment setup complete.")

DAY 4: AGENT ORCHESTRATION WITH LANGGRAPH
Started at: 2026-09-03T06:35:37.050651

Checking and installing required packages...
----------------------------------------
Already installed: langgraph
Already installed: langchain
Already installed: langchain-core
Installing: langchain-community
Installing: wikipedia-api
Already installed: requests
Already installed: pydantic
Already installed: typing_extensions
Already installed: gradio

----------------------------------------
All packages are ready.

Day 4 environment setup complete.


In [2]:
# CELL 2: Day 4 - Introduction to Agent Orchestration
# This cell provides an overview of what we will build today

print("DAY 4: AGENT ORCHESTRATION WITH LANGGRAPH")
print("=" * 60)
print()

print("TODAY'S OBJECTIVES:")
print("-" * 40)
print("1. Understand LangGraph State Management")
print("2. Create Agent Nodes (Research, Analysis, Writing)")
print("3. Build Supervisor Node with Routing Logic")
print("4. Implement Human-in-the-Loop (HITL)")
print("5. Create Graph-Based Orchestration")
print("6. Compare Different Orchestration Patterns")
print()

print("WHAT WE WILL BUILD:")
print("-" * 40)
print("We will create an orchestration system where:")
print("  - Each agent is a node in the graph")
print("  - Supervisor node routes tasks to appropriate agents")
print("  - State flows between nodes")
print("  - Human can approve or reject actions")
print()

print("LANGGRAPH CONCEPTS:")
print("-" * 40)
print("1. StateGraph: Graph-based workflow manager")
print("2. State: Typed dictionary that flows through nodes")
print("3. Nodes: Functions that process state")
print("4. Edges: Connections between nodes")
print("5. Conditional Edges: Routing based on state")
print("6. Checkpoints: Save and resume state")
print()

print("ORCHESTRATION PATTERNS:")
print("-" * 40)
print("1. Supervisor Pattern:")
print("   - Central supervisor delegates tasks")
print("   - Supervisor decides which agent to call next")
print("   - Agents report back to supervisor")
print()
print("2. Swarm Pattern:")
print("   - No central control")
print("   - Agents work independently")
print("   - Agents share information directly")
print()
print("3. Hierarchical Pattern:")
print("   - Multiple levels of supervision")
print("   - Sub-supervisors manage groups")
print("   - Complex workflows with delegation")
print()
print("4. Human-in-the-Loop Pattern:")
print("   - Human approval at key decision points")
print("   - Review and correction of outputs")
print("   - Manual intervention when needed")
print()

print("TOOLS WE WILL USE:")
print("-" * 40)
print("1. LangGraph - Graph-based orchestration")
print("2. TypedDict - Type-safe state management")
print("3. All tools from previous days")
print("4. Gradio - Interactive UI")
print()

print("EXPECTED OUTPUTS:")
print("-" * 40)
print("1. Complete LangGraph orchestration system")
print("2. Supervisor, Swarm, and HITL patterns")
print("3. Interactive UI for testing")
print("4. Performance comparison of patterns")
print()

print("HOW THIS BUILDS ON PREVIOUS DAYS:")
print("-" * 40)
print("Day 1: Single agent with tools")
print("Day 2: Single agent with planning and memory")
print("Day 3: Multiple agents collaborating")
print("Day 4: Advanced orchestration with LangGraph (Today)")
print("Day 5: Complete production system")
print()

print("SAMPLE QUERIES WE WILL HANDLE TODAY:")
print("-" * 40)
print("1. 'Research artificial intelligence'")
print("   -> Supervisor routes to ResearchAgent")
print("2. 'Analyze machine learning trends'")
print("   -> Supervisor routes to AnalysisAgent")
print("3. 'Create report on quantum computing'")
print("   -> Multiple agents in sequence")
print("4. Complex multi-step queries")
print("   -> Full orchestration with HITL")
print()

print("=" * 60)
print("Ready to start building Day 4!")
print("Run CELL 3 to create the base tools and agents.")

DAY 4: AGENT ORCHESTRATION WITH LANGGRAPH

TODAY'S OBJECTIVES:
----------------------------------------
1. Understand LangGraph State Management
2. Create Agent Nodes (Research, Analysis, Writing)
3. Build Supervisor Node with Routing Logic
4. Implement Human-in-the-Loop (HITL)
5. Create Graph-Based Orchestration
6. Compare Different Orchestration Patterns

WHAT WE WILL BUILD:
----------------------------------------
We will create an orchestration system where:
  - Each agent is a node in the graph
  - Supervisor node routes tasks to appropriate agents
  - State flows between nodes
  - Human can approve or reject actions

LANGGRAPH CONCEPTS:
----------------------------------------
1. StateGraph: Graph-based workflow manager
2. State: Typed dictionary that flows through nodes
3. Nodes: Functions that process state
4. Edges: Connections between nodes
5. Conditional Edges: Routing based on state
6. Checkpoints: Save and resume state

ORCHESTRATION PATTERNS:
-----------------------------

In [3]:
# CELL 3: Base Tools and Agent Classes for Day 4
# This cell creates the base tools and agent classes needed for orchestration

import re
import math
import json
import time
import requests
import wikipediaapi
from typing import Dict, Any, Optional, List, Tuple, TypedDict, Annotated
from datetime import datetime
from abc import ABC, abstractmethod

print("Building Base Tools and Agent Classes...")
print("=" * 60)

# ============================================================================
# TOOL IMPLEMENTATIONS
# ============================================================================

class CalculatorTool:
    name = "calculator"
    description = "Perform mathematical calculations"
    
    def _run(self, expression: str) -> str:
        try:
            expression = expression.strip()
            if not expression:
                return "Error: No expression provided"
            
            safe_context = {
                '__builtins__': {},
                'math': math,
                'sqrt': math.sqrt,
                'sin': math.sin,
                'cos': math.cos,
                'tan': math.tan,
                'log': math.log,
                'log10': math.log10,
                'abs': abs,
                'ceil': math.ceil,
                'floor': math.floor,
                'round': round,
                'pi': math.pi,
                'e': math.e
            }
            
            result = eval(expression, safe_context)
            return f"Result: {float(result)}"
        except Exception as e:
            return f"Error: {str(e)}"

class WikipediaTool:
    name = "wikipedia_search"
    description = "Search Wikipedia for information"
    
    def __init__(self):
        self._wiki = wikipediaapi.Wikipedia(
            language='en',
            user_agent='Orchestration-Project/1.0'
        )
        self._cache = {}
    
    def _clean_text(self, text: str, max_length: int = 500) -> str:
        if not text:
            return "No content available."
        text = re.sub(r'\s+', ' ', text)
        if len(text) > max_length:
            text = text[:max_length] + "..."
        return text.strip()
    
    def _run(self, query: str, max_results: int = 2) -> str:
        try:
            page = self._wiki.page(query)
            if page.exists():
                return f"Wikipedia Article: {page.title}\n\n{self._clean_text(page.summary, 500)}\n\nURL: {page.fullurl}"
            
            results = list(self._wiki.search(query))[:max_results]
            if not results:
                return f"No Wikipedia results found for: {query}"
            
            output = f"Wikipedia Search Results for '{query}':\n\n"
            for idx, title in enumerate(results, 1):
                page = self._wiki.page(title)
                if page.exists():
                    output += f"{idx}. {title}\n"
                    output += f"   {self._clean_text(page.summary, 200)}\n"
                    output += f"   URL: {page.fullurl}\n\n"
            return output
        except Exception as e:
            return f"Error searching Wikipedia: {str(e)}"

class WebSearchTool:
    name = "web_search"
    description = "Search the web for information"
    
    def _run(self, query: str, max_results: int = 2) -> str:
        try:
            url = "https://en.wikipedia.org/w/api.php"
            params = {
                "action": "query",
                "list": "search",
                "srsearch": query,
                "format": "json",
                "srlimit": max_results,
                "srprop": "snippet"
            }
            headers = {"User-Agent": "Orchestration-Project/1.0"}
            response = requests.get(url, params=params, headers=headers, timeout=10)
            
            if response.status_code != 200:
                return f"No results found for: {query}"
            
            data = response.json()
            results = data.get("query", {}).get("search", [])
            
            if not results:
                return f"No results found for: {query}"
            
            output = f"Search Results for '{query}':\n\n"
            for idx, item in enumerate(results, 1):
                title = item.get("title", "")
                snippet = re.sub(r'<[^>]+>', '', item.get("snippet", ""))
                output += f"{idx}. {title}\n"
                output += f"   {snippet[:200]}...\n\n"
            return output
        except Exception as e:
            return f"Error performing search: {str(e)}"

# ============================================================================
# TOOL REGISTRY
# ============================================================================

class ToolRegistry:
    def __init__(self):
        self._tools = {}
    
    def register_tool(self, tool):
        self._tools[tool.name] = tool
        print(f"  Registered: {tool.name}")
    
    def get_tool(self, tool_name):
        return self._tools.get(tool_name)
    
    def list_tools(self):
        return list(self._tools.keys())
    
    def execute_tool(self, tool_name, **kwargs):
        tool = self.get_tool(tool_name)
        if not tool:
            return {"success": False, "error": f"Tool '{tool_name}' not found"}
        try:
            result = tool._run(**kwargs)
            return {"success": True, "result": result}
        except Exception as e:
            return {"success": False, "error": str(e)}

# ============================================================================
# BASE AGENT CLASS
# ============================================================================

class BaseAgent(ABC):
    def __init__(self, name: str, role: str, goal: str, registry: ToolRegistry):
        self.name = name
        self.role = role
        self.goal = goal
        self.registry = registry
        self.memory = []
        self.output = None
    
    @abstractmethod
    def execute(self, input_data: Any) -> Any:
        pass
    
    def get_tool(self, tool_name: str):
        return self.registry.get_tool(tool_name)
    
    def execute_tool(self, tool_name: str, **kwargs):
        return self.registry.execute_tool(tool_name, **kwargs)
    
    def store_memory(self, content: str):
        self.memory.append({
            "timestamp": datetime.now().isoformat(),
            "content": content
        })

# ============================================================================
# BUILD TOOLS AND REGISTRY
# ============================================================================

print("\nCreating tool registry...")
registry = ToolRegistry()

print("\nRegistering tools:")
registry.register_tool(WikipediaTool())
registry.register_tool(WebSearchTool())
registry.register_tool(CalculatorTool())

print(f"\nTools available: {registry.list_tools()}")

print("\n" + "=" * 60)
print("Base tools and registry created successfully.")

Building Base Tools and Agent Classes...

Creating tool registry...

Registering tools:
  Registered: wikipedia_search
  Registered: web_search
  Registered: calculator

Tools available: ['wikipedia_search', 'web_search', 'calculator']

Base tools and registry created successfully.


In [4]:
# CELL 4: Specialized Agents for Orchestration
# This cell creates specialized agents that will be used as nodes in the graph

import json
from typing import Dict, Any, Optional, List
from datetime import datetime

print("Creating Specialized Agents for Orchestration...")
print("=" * 60)

# ============================================================================
# RESEARCH AGENT
# ============================================================================

class ResearchAgent(BaseAgent):
    def __init__(self, registry: ToolRegistry):
        super().__init__(
            name="ResearchAgent",
            role="Information Gatherer",
            goal="Find and gather relevant information from multiple sources",
            registry=registry
        )
        self.sources_used = []
    
    def execute(self, input_data: Any) -> Dict[str, Any]:
        """
        Execute research on a given topic.
        
        Args:
            input_data: Can be string (topic) or dict with 'query' field
            
        Returns:
            Dictionary containing research results
        """
        if isinstance(input_data, str):
            topic = input_data
        elif isinstance(input_data, dict):
            topic = input_data.get('query', input_data.get('topic', str(input_data)))
        else:
            topic = str(input_data)
        
        print(f"\n[{self.name}] Researching: {topic}")
        print("-" * 40)
        
        results = {
            "topic": topic,
            "sources": [],
            "information": [],
            "timestamp": datetime.now().isoformat()
        }
        
        # Try Wikipedia
        print("  Searching Wikipedia...")
        wiki_result = self.execute_tool("wikipedia_search", query=topic)
        if wiki_result.get('success') and wiki_result.get('result'):
            if "No Wikipedia" not in wiki_result['result']:
                results["information"].append({
                    "source": "Wikipedia",
                    "content": wiki_result['result'],
                    "timestamp": datetime.now().isoformat()
                })
                results["sources"].append("Wikipedia")
                self.sources_used.append("Wikipedia")
                print("  Found Wikipedia information")
        
        # Try Web Search
        print("  Searching Web...")
        web_result = self.execute_tool("web_search", query=topic)
        if web_result.get('success') and web_result.get('result'):
            if "No results" not in web_result['result']:
                results["information"].append({
                    "source": "Web Search",
                    "content": web_result['result'],
                    "timestamp": datetime.now().isoformat()
                })
                results["sources"].append("Web Search")
                self.sources_used.append("Web Search")
                print("  Found Web information")
        
        # If no results, provide suggestions
        if not results["information"]:
            results["information"].append({
                "source": "System",
                "content": f"No specific information found for '{topic}'.\n\nSuggestions:\n1. Try a simpler query (e.g., 'AI' instead of 'artificial intelligence and its applications')\n2. Check the spelling of your query\n3. Try a different topic\n\nCommon topics that work well:\n- Artificial Intelligence\n- Machine Learning\n- Python\n- Quantum Computing",
                "timestamp": datetime.now().isoformat()
            })
        
        # Generate summary
        results["summary"] = self._generate_summary(results["information"])
        self.output = results
        
        print(f"  Research complete. Found {len(results['information'])} sources.")
        return results
    
    def _generate_summary(self, information: List[Dict]) -> str:
        if not information:
            return "No information available."
        
        summary = f"Research found {len(information)} sources:\n"
        for idx, info in enumerate(information, 1):
            source = info.get('source', 'Unknown')
            content = info.get('content', '')
            summary += f"{idx}. {source}: {content[:150]}...\n"
        
        return summary

# ============================================================================
# ANALYSIS AGENT
# ============================================================================

class AnalysisAgent(BaseAgent):
    def __init__(self, registry: ToolRegistry):
        super().__init__(
            name="AnalysisAgent",
            role="Information Analyst",
            goal="Analyze and synthesize information into structured insights",
            registry=registry
        )
    
    def execute(self, input_data: Any) -> Dict[str, Any]:
        """
        Execute analysis on research results.
        
        Args:
            input_data: Research results from ResearchAgent
            
        Returns:
            Dictionary containing analysis results
        """
        # Handle different input types
        if isinstance(input_data, dict):
            if 'topic' in input_data:
                topic = input_data.get('topic', 'Unknown')
                information = input_data.get('information', [])
            else:
                topic = input_data.get('query', 'Unknown')
                information = input_data.get('information', [])
        else:
            topic = str(input_data)
            information = []
        
        print(f"\n[{self.name}] Analyzing: {topic}")
        print("-" * 40)
        
        analysis = {
            "topic": topic,
            "key_findings": [],
            "insights": [],
            "recommendations": [],
            "timestamp": datetime.now().isoformat()
        }
        
        # Process each piece of information
        for info in information:
            content = info.get('content', '')
            source = info.get('source', 'Unknown')
            
            # Extract key points
            key_points = self._extract_key_points(content, source)
            if key_points:
                analysis["key_findings"].extend(key_points)
            
            # Generate insights
            insights = self._generate_insights(content, topic)
            if insights:
                analysis["insights"].extend(insights)
        
        # Remove duplicates
        analysis["key_findings"] = list(dict.fromkeys(analysis["key_findings"]))
        analysis["insights"] = list(dict.fromkeys(analysis["insights"]))
        
        # Generate recommendations
        analysis["recommendations"] = self._generate_recommendations(analysis)
        
        self.output = analysis
        
        print(f"  Analysis complete. Found {len(analysis['key_findings'])} key findings.")
        print(f"  Generated {len(analysis['insights'])} insights.")
        return analysis
    
    def _extract_key_points(self, content: str, source: str) -> List[str]:
        points = []
        lines = content.split('\n')
        for line in lines:
            line = line.strip()
            if line and (line.startswith('•') or line.startswith('-') or 
                         line.startswith('1.') or line.startswith('2.') or
                         line.startswith('3.') or line.startswith('4.') or
                         line.startswith('5.') or line.startswith('*')):
                clean_line = re.sub(r'^[•\-*\d.]+', '', line).strip()
                if clean_line and len(clean_line) > 10:
                    points.append(clean_line[:200])
        
        if not points:
            sentences = re.split(r'[.!?]', content)
            for sentence in sentences[:5]:
                sentence = sentence.strip()
                if 20 < len(sentence) < 200:
                    points.append(sentence)
        
        return points[:5]
    
    def _generate_insights(self, content: str, topic: str) -> List[str]:
        insights = []
        content_lower = content.lower()
        
        if "AI" in content or "artificial intelligence" in content_lower:
            if "machine learning" in content_lower:
                insights.append(f"Machine learning is closely related to {topic}")
        
        if "data" in content_lower:
            insights.append(f"Data plays a crucial role in {topic}")
        
        if "model" in content_lower or "algorithm" in content_lower:
            insights.append(f"Algorithms and models are fundamental to {topic}")
        
        if "future" in content_lower or "development" in content_lower:
            insights.append(f"{topic} is an evolving field with future developments")
        
        if not insights and len(content) > 100:
            insights.append(f"{topic} is a significant and complex subject with multiple aspects")
        
        return insights
    
    def _generate_recommendations(self, analysis: Dict) -> List[str]:
        recommendations = []
        
        if analysis["key_findings"]:
            recommendations.append("Focus on the key findings identified")
        
        if analysis["insights"]:
            recommendations.append("Consider the insights for deeper understanding")
        
        if len(analysis["key_findings"]) < 2:
            recommendations.append("Conduct additional research on this topic")
        
        recommendations.append("Document findings for future reference")
        
        return recommendations

# ============================================================================
# WRITING AGENT
# ============================================================================

class WritingAgent(BaseAgent):
    def __init__(self, registry: ToolRegistry):
        super().__init__(
            name="WritingAgent",
            role="Report Writer",
            goal="Create well-structured, professional reports and documentation",
            registry=registry
        )
    
    def execute(self, input_data: Any) -> Dict[str, Any]:
        """
        Execute report writing based on analysis results.
        
        Args:
            input_data: Analysis results from AnalysisAgent
            
        Returns:
            Dictionary containing the final report
        """
        # Handle different input types
        if isinstance(input_data, dict):
            topic = input_data.get('topic', 'Unknown')
            findings = input_data.get('key_findings', [])
            insights = input_data.get('insights', [])
            recommendations = input_data.get('recommendations', [])
        else:
            topic = str(input_data)
            findings = []
            insights = []
            recommendations = []
        
        print(f"\n[{self.name}] Writing report on: {topic}")
        print("-" * 40)
        
        report = {
            "topic": topic,
            "title": self._generate_title(topic),
            "executive_summary": self._generate_executive_summary(topic, findings),
            "findings": findings[:10],
            "insights": insights[:5],
            "recommendations": recommendations[:5],
            "timestamp": datetime.now().isoformat()
        }
        
        # Generate full report
        report["full_report"] = self._generate_full_report(report)
        
        self.output = report
        
        print(f"  Report complete. Length: {len(report['full_report'])} characters.")
        return report
    
    def _generate_title(self, topic: str) -> str:
        titles = [
            f"Comprehensive Report on {topic}",
            f"Analysis and Insights: {topic}",
            f"Research Findings: {topic}"
        ]
        return titles[hash(topic) % len(titles)]
    
    def _generate_executive_summary(self, topic: str, findings: List[str]) -> str:
        if not findings:
            return f"No significant findings discovered for {topic}."
        
        summary = f"This report provides an analysis of {topic}. "
        if len(findings) >= 3:
            summary += f"Key findings include: {', '.join(findings[:3])}. "
        else:
            summary += f"Key findings include: {'; '.join(findings)}. "
        summary += "This analysis is based on information from multiple sources."
        
        return summary
    
    def _generate_full_report(self, report: Dict) -> str:
        lines = []
        
        lines.append("=" * 60)
        lines.append(report['title'])
        lines.append("=" * 60)
        lines.append("")
        lines.append(f"Generated: {report['timestamp']}")
        lines.append("")
        
        lines.append("EXECUTIVE SUMMARY")
        lines.append("-" * 40)
        lines.append(report['executive_summary'])
        lines.append("")
        
        lines.append("KEY FINDINGS")
        lines.append("-" * 40)
        if report['findings']:
            for i, finding in enumerate(report['findings'], 1):
                lines.append(f"{i}. {finding}")
        else:
            lines.append("No specific findings were identified.")
        lines.append("")
        
        lines.append("INSIGHTS")
        lines.append("-" * 40)
        if report['insights']:
            for insight in report['insights']:
                lines.append(f"  - {insight}")
        else:
            lines.append("No specific insights were generated.")
        lines.append("")
        
        lines.append("RECOMMENDATIONS")
        lines.append("-" * 40)
        if report['recommendations']:
            for i, rec in enumerate(report['recommendations'], 1):
                lines.append(f"{i}. {rec}")
        else:
            lines.append("No recommendations available.")
        lines.append("")
        
        lines.append("=" * 60)
        lines.append("END OF REPORT")
        lines.append("=" * 60)
        
        return "\n".join(lines)

# ============================================================================
# CREATE AGENTS
# ============================================================================

print("\nCreating specialized agents...")
research_agent = ResearchAgent(registry)
analysis_agent = AnalysisAgent(registry)
writing_agent = WritingAgent(registry)

print(f"\nAgents created:")
print(f"  - {research_agent.name} ({research_agent.role})")
print(f"  - {analysis_agent.name} ({analysis_agent.role})")
print(f"  - {writing_agent.name} ({writing_agent.role})")

print("\n" + "=" * 60)
print("Specialized agents created successfully.")

Creating Specialized Agents for Orchestration...

Creating specialized agents...

Agents created:
  - ResearchAgent (Information Gatherer)
  - AnalysisAgent (Information Analyst)
  - WritingAgent (Report Writer)

Specialized agents created successfully.


In [5]:
# CELL 5: LangGraph State Definition
# This cell defines the state structure for LangGraph orchestration

from typing import Dict, Any, Optional, List, TypedDict, Annotated
from datetime import datetime
import operator

print("Defining LangGraph State Structure...")
print("=" * 60)

# ============================================================================
# STATE DEFINITION
# ============================================================================

class OrchestrationState(TypedDict):
    """
    State structure for the LangGraph orchestration.
    This state flows through all nodes in the graph.
    """
    query: str
    current_node: str
    iteration: int
    max_iterations: int
    
    # Research results
    research_results: Optional[Dict[str, Any]]
    analysis_results: Optional[Dict[str, Any]]
    writing_results: Optional[Dict[str, Any]]
    
    # Agent outputs
    agent_outputs: Annotated[List[Dict[str, Any]], operator.add]
    
    # Status tracking
    status: str
    errors: Annotated[List[str], operator.add]
    messages: Annotated[List[str], operator.add]
    
    # Human-in-the-loop
    requires_approval: bool
    approved: bool
    human_feedback: Optional[str]
    
    # Final output
    final_response: Optional[str]
    timestamp: str

def create_initial_state(query: str) -> OrchestrationState:
    """
    Create an initial state for the orchestration.
    
    Args:
        query: The user query
        
    Returns:
        Initial OrchestrationState
    """
    return {
        "query": query,
        "current_node": "start",
        "iteration": 0,
        "max_iterations": 5,
        "research_results": None,
        "analysis_results": None,
        "writing_results": None,
        "agent_outputs": [],
        "status": "initialized",
        "errors": [],
        "messages": [f"Starting orchestration for: {query}"],
        "requires_approval": False,
        "approved": False,
        "human_feedback": None,
        "final_response": None,
        "timestamp": datetime.now().isoformat()
    }

def update_state(state: OrchestrationState, **kwargs) -> OrchestrationState:
    """
    Update the state with new values.
    
    Args:
        state: Current state
        **kwargs: Values to update
        
    Returns:
        Updated state
    """
    updated_state = dict(state)
    for key, value in kwargs.items():
        if key in updated_state:
            updated_state[key] = value
    return updated_state

def add_message(state: OrchestrationState, message: str) -> OrchestrationState:
    """
    Add a message to the state.
    
    Args:
        state: Current state
        message: Message to add
        
    Returns:
        Updated state
    """
    messages = list(state.get('messages', []))
    messages.append(message)
    return update_state(state, messages=messages)

def add_error(state: OrchestrationState, error: str) -> OrchestrationState:
    """
    Add an error to the state.
    
    Args:
        state: Current state
        error: Error message
        
    Returns:
        Updated state
    """
    errors = list(state.get('errors', []))
    errors.append(error)
    return update_state(state, errors=errors, status="error")

def add_agent_output(state: OrchestrationState, agent_name: str, output: Any) -> OrchestrationState:
    """
    Add an agent output to the state.
    
    Args:
        state: Current state
        agent_name: Name of the agent
        output: Agent output
        
    Returns:
        Updated state
    """
    outputs = list(state.get('agent_outputs', []))
    outputs.append({
        "agent": agent_name,
        "output": output,
        "timestamp": datetime.now().isoformat()
    })
    return update_state(state, agent_outputs=outputs)

print("State structure defined:")
print(f"  - State fields: {list(OrchestrationState.__annotations__.keys())}")
print(f"  - Create initial state: create_initial_state()")
print(f"  - Update state: update_state()")
print(f"  - Add message: add_message()")
print(f"  - Add error: add_error()")
print(f"  - Add agent output: add_agent_output()")

print("\n" + "=" * 60)
print("State definition complete.")

Defining LangGraph State Structure...
State structure defined:
  - State fields: ['query', 'current_node', 'iteration', 'max_iterations', 'research_results', 'analysis_results', 'writing_results', 'agent_outputs', 'status', 'errors', 'messages', 'requires_approval', 'approved', 'human_feedback', 'final_response', 'timestamp']
  - Create initial state: create_initial_state()
  - Update state: update_state()
  - Add message: add_message()
  - Add error: add_error()
  - Add agent output: add_agent_output()

State definition complete.


In [6]:
# CELL 6: LangGraph Nodes
# This cell creates the node functions for the LangGraph orchestration

from typing import Dict, Any, Optional
import time

print("Creating LangGraph Nodes...")
print("=" * 60)

# ============================================================================
# NODE FUNCTIONS
# ============================================================================

def research_node(state: OrchestrationState) -> OrchestrationState:
    """
    Research node - executes the ResearchAgent.
    
    Args:
        state: Current orchestration state
        
    Returns:
        Updated state
    """
    print(f"\n[Node: Research] Processing...")
    
    query = state.get('query', '')
    iteration = state.get('iteration', 0)
    
    # Check if we've exceeded max iterations
    if iteration >= state.get('max_iterations', 5):
        return add_error(state, "Max iterations exceeded in research node")
    
    try:
        # Execute the research agent
        print(f"  Researching: {query}")
        result = research_agent.execute(query)
        
        # Update state
        new_state = update_state(
            state,
            research_results=result,
            current_node="research",
            iteration=iteration + 1,
            status="research_complete"
        )
        
        new_state = add_message(new_state, f"Research completed for: {query}")
        new_state = add_agent_output(new_state, "ResearchAgent", result)
        
        print(f"  Research complete. Found {len(result.get('information', []))} sources.")
        return new_state
        
    except Exception as e:
        error_msg = f"Research node error: {str(e)}"
        print(f"  Error: {error_msg}")
        return add_error(state, error_msg)

def analysis_node(state: OrchestrationState) -> OrchestrationState:
    """
    Analysis node - executes the AnalysisAgent.
    
    Args:
        state: Current orchestration state
        
    Returns:
        Updated state
    """
    print(f"\n[Node: Analysis] Processing...")
    
    research_results = state.get('research_results')
    if not research_results:
        return add_error(state, "No research results available for analysis")
    
    iteration = state.get('iteration', 0)
    if iteration >= state.get('max_iterations', 5):
        return add_error(state, "Max iterations exceeded in analysis node")
    
    try:
        print(f"  Analyzing research data...")
        result = analysis_agent.execute(research_results)
        
        new_state = update_state(
            state,
            analysis_results=result,
            current_node="analysis",
            iteration=iteration + 1,
            status="analysis_complete"
        )
        
        new_state = add_message(new_state, "Analysis completed")
        new_state = add_agent_output(new_state, "AnalysisAgent", result)
        
        print(f"  Analysis complete. Found {len(result.get('key_findings', []))} findings.")
        return new_state
        
    except Exception as e:
        error_msg = f"Analysis node error: {str(e)}"
        print(f"  Error: {error_msg}")
        return add_error(state, error_msg)

def writing_node(state: OrchestrationState) -> OrchestrationState:
    """
    Writing node - executes the WritingAgent.
    
    Args:
        state: Current orchestration state
        
    Returns:
        Updated state
    """
    print(f"\n[Node: Writing] Processing...")
    
    analysis_results = state.get('analysis_results')
    if not analysis_results:
        return add_error(state, "No analysis results available for writing")
    
    iteration = state.get('iteration', 0)
    if iteration >= state.get('max_iterations', 5):
        return add_error(state, "Max iterations exceeded in writing node")
    
    try:
        print(f"  Writing report...")
        result = writing_agent.execute(analysis_results)
        
        new_state = update_state(
            state,
            writing_results=result,
            current_node="writing",
            iteration=iteration + 1,
            status="writing_complete"
        )
        
        new_state = add_message(new_state, "Writing completed")
        new_state = add_agent_output(new_state, "WritingAgent", result)
        
        print(f"  Writing complete. Report length: {len(result.get('full_report', ''))} characters.")
        return new_state
        
    except Exception as e:
        error_msg = f"Writing node error: {str(e)}"
        print(f"  Error: {error_msg}")
        return add_error(state, error_msg)

def supervisor_node(state: OrchestrationState) -> OrchestrationState:
    """
    Supervisor node - routes tasks to appropriate agents.
    
    Args:
        state: Current orchestration state
        
    Returns:
        Updated state
    """
    print(f"\n[Node: Supervisor] Processing...")
    
    query = state.get('query', '')
    iteration = state.get('iteration', 0)
    
    if iteration >= state.get('max_iterations', 5):
        return add_error(state, "Max iterations exceeded in supervisor node")
    
    # Determine which agents to run based on query
    query_lower = query.lower()
    tasks = []
    
    # Always include research
    tasks.append("research")
    
    # Add analysis for complex queries
    if len(query.split()) > 5 or "analyze" in query_lower or "explain" in query_lower:
        tasks.append("analysis")
    
    # Add writing for report queries
    if "report" in query_lower or "summary" in query_lower or "write" in query_lower:
        tasks.append("writing")
    
    # If no specific tasks, add writing to summarize
    if len(tasks) == 1:
        tasks.append("writing")
    
    print(f"  Planning tasks: {tasks}")
    
    new_state = update_state(
        state,
        current_node="supervisor",
        iteration=iteration + 1,
        status="planning_complete"
    )
    
    new_state = add_message(new_state, f"Supervisor planned tasks: {tasks}")
    new_state = add_agent_output(new_state, "SupervisorAgent", {"planned_tasks": tasks})
    
    return new_state

# ============================================================================
# CONDITIONAL ROUTING FUNCTIONS
# ============================================================================

def route_after_supervisor(state: OrchestrationState) -> str:
    """
    Determine which node to go to after supervisor.
    
    Args:
        state: Current state
        
    Returns:
        Name of the next node
    """
    query = state.get('query', '')
    iteration = state.get('iteration', 0)
    
    # Check if we should stop
    if state.get('status') == 'error':
        return "error"
    
    if iteration >= state.get('max_iterations', 5):
        return "finalize"
    
    # Check what's been done
    has_research = state.get('research_results') is not None
    has_analysis = state.get('analysis_results') is not None
    has_writing = state.get('writing_results') is not None
    
    # Determine next step
    if not has_research:
        return "research"
    elif not has_analysis and "analysis" in query.lower():
        return "analysis"
    elif not has_writing:
        return "writing"
    else:
        return "finalize"

def route_after_agent(state: OrchestrationState) -> str:
    """
    Determine which node to go to after an agent completes.
    
    Args:
        state: Current state
        
    Returns:
        Name of the next node
    """
    if state.get('status') == 'error':
        return "error"
    
    current_node = state.get('current_node', '')
    
    if current_node == "research":
        # Check if we need analysis
        query = state.get('query', '')
        if "analyze" in query.lower() or "explain" in query.lower():
            return "analysis"
        else:
            return "writing"
    elif current_node == "analysis":
        return "writing"
    elif current_node == "writing":
        return "finalize"
    else:
        return "finalize"

def route_to_hitl(state: OrchestrationState) -> bool:
    """
    Determine if we need human-in-the-loop approval.
    
    Args:
        state: Current state
        
    Returns:
        True if HITL is needed
    """
    # Check if we need approval for certain operations
    if state.get('requires_approval', False):
        return True
    
    # Check if there are errors
    if len(state.get('errors', [])) > 0:
        return True
    
    # Check if this is a complex query
    query = state.get('query', '')
    if len(query.split()) > 10:
        return True
    
    return False

# ============================================================================
# ERROR HANDLING NODE
# ============================================================================

def error_node(state: OrchestrationState) -> OrchestrationState:
    """
    Error handling node.
    
    Args:
        state: Current state
        
    Returns:
        Updated state
    """
    print(f"\n[Node: Error Handler] Processing...")
    
    errors = state.get('errors', [])
    error_msg = errors[-1] if errors else "Unknown error"
    
    print(f"  Error: {error_msg}")
    
    new_state = update_state(
        state,
        current_node="error",
        status="error",
        final_response=f"Error occurred: {error_msg}"
    )
    
    new_state = add_message(new_state, f"Error handled: {error_msg}")
    
    return new_state

def finalize_node(state: OrchestrationState) -> OrchestrationState:
    """
    Finalize node - prepares the final response.
    
    Args:
        state: Current state
        
    Returns:
        Updated state with final response
    """
    print(f"\n[Node: Finalize] Processing...")
    
    # Try to get final response from writing results
    writing_results = state.get('writing_results')
    if writing_results and isinstance(writing_results, dict):
        full_report = writing_results.get('full_report', '')
        if full_report:
            final_response = full_report
            print("  Using full report as final response")
        else:
            final_response = writing_results.get('executive_summary', 'Report generated successfully.')
            print("  Using executive summary as final response")
    else:
        # Try to build response from analysis
        analysis_results = state.get('analysis_results')
        if analysis_results and isinstance(analysis_results, dict):
            findings = analysis_results.get('key_findings', [])
            if findings:
                final_response = f"Analysis completed with {len(findings)} findings:\n\n"
                for i, finding in enumerate(findings[:5], 1):
                    final_response += f"{i}. {finding}\n"
            else:
                final_response = "Analysis completed but no specific findings were identified."
        else:
            final_response = f"Query processed: {state.get('query', 'Unknown')}"
    
    # Add summary
    summary = []
    summary.append("=" * 60)
    summary.append("ORCHESTRATION COMPLETE")
    summary.append("=" * 60)
    summary.append(f"Query: {state.get('query', '')}")
    summary.append(f"Status: {state.get('status', 'unknown')}")
    summary.append(f"Iterations: {state.get('iteration', 0)}")
    summary.append("")
    summary.append("Agents executed:")
    
    for output in state.get('agent_outputs', []):
        agent = output.get('agent', 'Unknown')
        summary.append(f"  - {agent}")
    
    summary.append("")
    summary.append("=" * 60)
    summary.append(final_response)
    summary.append("=" * 60)
    
    final_response_with_summary = "\n".join(summary)
    
    new_state = update_state(
        state,
        current_node="finalize",
        status="complete",
        final_response=final_response_with_summary
    )
    
    new_state = add_message(new_state, "Orchestration complete")
    
    print(f"  Finalized. Response length: {len(final_response_with_summary)} characters.")
    return new_state

print("\nNodes created:")
print("  - research_node: Executes ResearchAgent")
print("  - analysis_node: Executes AnalysisAgent")
print("  - writing_node: Executes WritingAgent")
print("  - supervisor_node: Plans and routes tasks")
print("  - error_node: Handles errors")
print("  - finalize_node: Prepares final response")
print("")
print("Routing functions:")
print("  - route_after_supervisor: Routes after supervisor")
print("  - route_after_agent: Routes after agent execution")
print("  - route_to_hitl: Determines if HITL is needed")

print("\n" + "=" * 60)
print("LangGraph nodes created successfully.")

Creating LangGraph Nodes...

Nodes created:
  - research_node: Executes ResearchAgent
  - analysis_node: Executes AnalysisAgent
  - writing_node: Executes WritingAgent
  - supervisor_node: Plans and routes tasks
  - error_node: Handles errors
  - finalize_node: Prepares final response

Routing functions:
  - route_after_supervisor: Routes after supervisor
  - route_after_agent: Routes after agent execution
  - route_to_hitl: Determines if HITL is needed

LangGraph nodes created successfully.


In [8]:
# CELL 7: LangGraph Graph Construction (Fixed)
# This cell builds the complete LangGraph orchestration graph

from langgraph.graph import StateGraph, END
import time

# Try different import paths for MemorySaver
try:
    from langgraph.checkpoint.memory import MemorySaver
    print("MemorySaver imported from langgraph.checkpoint.memory")
except ImportError:
    try:
        from langgraph.memory import MemorySaver
        print("MemorySaver imported from langgraph.memory")
    except ImportError:
        # Create a simple memory saver if not available
        print("MemorySaver not available, using custom memory")
        MemorySaver = None

print("Building LangGraph Orchestration Graph...")
print("=" * 60)

# ============================================================================
# CUSTOM MEMORY SAVER (if LangGraph MemorySaver not available)
# ============================================================================

class SimpleMemorySaver:
    """
    Simple memory saver for checkpointing state.
    """
    def __init__(self):
        self.checkpoints = {}
    
    def save(self, key, state):
        self.checkpoints[key] = state
    
    def load(self, key):
        return self.checkpoints.get(key)

# ============================================================================
# BUILD THE GRAPH
# ============================================================================

def build_orchestration_graph():
    """
    Build the complete LangGraph orchestration graph.
    
    Returns:
        Compiled graph
    """
    print("\nBuilding graph...")
    
    # Create the graph
    graph = StateGraph(OrchestrationState)
    
    # Add nodes
    print("  Adding nodes...")
    graph.add_node("supervisor", supervisor_node)
    graph.add_node("research", research_node)
    graph.add_node("analysis", analysis_node)
    graph.add_node("writing", writing_node)
    graph.add_node("error", error_node)
    graph.add_node("finalize", finalize_node)
    
    # Add edges from supervisor
    print("  Adding edges...")
    graph.add_conditional_edges(
        "supervisor",
        route_after_supervisor,
        {
            "research": "research",
            "analysis": "analysis",
            "writing": "writing",
            "finalize": "finalize",
            "error": "error"
        }
    )
    
    # Add edges from research
    graph.add_conditional_edges(
        "research",
        route_after_agent,
        {
            "analysis": "analysis",
            "writing": "writing",
            "finalize": "finalize",
            "error": "error"
        }
    )
    
    # Add edges from analysis
    graph.add_conditional_edges(
        "analysis",
        route_after_agent,
        {
            "writing": "writing",
            "finalize": "finalize",
            "error": "error"
        }
    )
    
    # Add edges from writing
    graph.add_conditional_edges(
        "writing",
        route_after_agent,
        {
            "finalize": "finalize",
            "error": "error"
        }
    )
    
    # Add edges from error
    graph.add_edge("error", END)
    
    # Add edges from finalize
    graph.add_edge("finalize", END)
    
    # Set entry point
    print("  Setting entry point...")
    graph.set_entry_point("supervisor")
    
    # Compile with checkpointer if available
    print("  Compiling graph...")
    if MemorySaver:
        memory = MemorySaver()
        compiled_graph = graph.compile(checkpointer=memory)
    else:
        memory = SimpleMemorySaver()
        compiled_graph = graph.compile(checkpointer=memory)
    
    print("  Graph compiled successfully.")
    return compiled_graph

def visualize_graph(graph):
    """
    Print a simple visualization of the graph structure.
    
    Args:
        graph: The compiled graph
    """
    print("\nGraph Structure:")
    print("=" * 40)
    print("  START")
    print("    |")
    print("    v")
    print("  SUPERVISOR_NODE")
    print("    |")
    print("    +---> RESEARCH_NODE")
    print("    |       |")
    print("    |       +---> ANALYSIS_NODE")
    print("    |       |       |")
    print("    |       |       +---> WRITING_NODE")
    print("    |       |               |")
    print("    |       |               v")
    print("    +------>--------------> FINALIZE_NODE")
    print("                                |")
    print("                                v")
    print("                              ERROR_NODE")
    print("                                |")
    print("                                v")
    print("                               END")
    print("=" * 40)

# ============================================================================
# BUILD THE GRAPH
# ============================================================================

print("Building orchestration graph...")
orchestration_graph = build_orchestration_graph()

# Visualize the graph
visualize_graph(orchestration_graph)

print("\n" + "=" * 60)
print("LangGraph orchestration graph built successfully.")
print("Graph type:", type(orchestration_graph))

MemorySaver imported from langgraph.checkpoint.memory
Building LangGraph Orchestration Graph...
Building orchestration graph...

Building graph...
  Adding nodes...
  Adding edges...
  Setting entry point...
  Compiling graph...
  Graph compiled successfully.

Graph Structure:
  START
    |
    v
  SUPERVISOR_NODE
    |
    +---> RESEARCH_NODE
    |       |
    |       +---> ANALYSIS_NODE
    |       |       |
    |       |       +---> WRITING_NODE
    |       |               |
    |       |               v
    +------>--------------> FINALIZE_NODE
                                |
                                v
                              ERROR_NODE
                                |
                                v
                               END

LangGraph orchestration graph built successfully.
Graph type: <class 'langgraph.graph.state.CompiledStateGraph'>


In [9]:
# CELL 8: Human-in-the-Loop Implementation (Fixed)
# This cell implements the Human-in-the-Loop capability

from typing import Dict, Any, Optional
import time

print("Implementing Human-in-the-Loop...")
print("=" * 60)

# ============================================================================
# HITL NODES AND FUNCTIONS
# ============================================================================

class HumanInTheLoop:
    """
    Human-in-the-Loop manager for the orchestration system.
    """
    
    def __init__(self):
        self.approval_history = []
        self.pending_approvals = []
    
    def request_approval(self, state: OrchestrationState, node_name: str, 
                         output: Any) -> bool:
        """
        Request human approval for a node's output.
        
        Args:
            state: Current state
            node_name: Name of the node requesting approval
            output: Output to approve
            
        Returns:
            True if approved, False if rejected
        """
        query = state.get('query', '')
        iteration = state.get('iteration', 0)
        
        print(f"\n" + "=" * 60)
        print(f"HUMAN-IN-THE-LOOP APPROVAL REQUEST")
        print("=" * 60)
        print(f"Node: {node_name}")
        print(f"Query: {query}")
        print(f"Iteration: {iteration}")
        print("-" * 40)
        
        # Display output preview
        if isinstance(output, dict):
            if 'full_report' in output:
                preview = output['full_report'][:300] + "..."
            elif 'summary' in output:
                preview = output['summary'][:300] + "..."
            else:
                preview = str(output)[:300] + "..."
        else:
            preview = str(output)[:300] + "..."
        
        print(f"Output preview:")
        print(preview)
        print("-" * 40)
        
        # In a real system, this would be a UI prompt
        # For demo, auto-approve after showing
        print("Auto-approving in 2 seconds...")
        time.sleep(2)
        
        approved = True
        print(f"Approval: {'APPROVED' if approved else 'REJECTED'}")
        
        # Record in history
        self.approval_history.append({
            "node": node_name,
            "query": query,
            "approved": approved,
            "timestamp": datetime.now().isoformat()
        })
        
        return approved
    
    def add_approval_node(self, graph, node_name: str, node_func):
        """
        Add an approval node to the graph.
        
        Args:
            graph: The graph to add the node to
            node_name: Name of the node
            node_func: Function to execute
            
        Returns:
            Updated graph
        """
        def approval_wrapper(state: OrchestrationState) -> OrchestrationState:
            """Wrapper that adds HITL capability to a node."""
            print(f"\n[Node: {node_name}] Processing with HITL...")
            
            # Execute the node
            new_state = node_func(state)
            
            # Check if approval is needed
            if route_to_hitl(new_state):
                print("  Approval required...")
                # Get the output to approve
                output = None
                if node_name == "research":
                    output = new_state.get('research_results')
                elif node_name == "analysis":
                    output = new_state.get('analysis_results')
                elif node_name == "writing":
                    output = new_state.get('writing_results')
                elif node_name == "supervisor":
                    output = new_state.get('agent_outputs', [])
                
                if output:
                    approved = self.request_approval(new_state, node_name, output)
                    if approved:
                        new_state = add_message(new_state, f"{node_name} output approved")
                        new_state = update_state(new_state, approved=True)
                    else:
                        new_state = add_message(new_state, f"{node_name} output rejected")
                        new_state = add_error(new_state, f"{node_name} output rejected by human")
                else:
                    print("  No output to approve")
            else:
                print("  No approval needed")
            
            return new_state
        
        graph.add_node(node_name, approval_wrapper)
        return graph
    
    def get_approval_history(self) -> List[Dict]:
        """Get the approval history."""
        return self.approval_history
    
    def get_approval_summary(self) -> str:
        """Get a summary of approvals."""
        if not self.approval_history:
            return "No approvals requested."
        
        lines = []
        lines.append("=" * 60)
        lines.append("APPROVAL HISTORY")
        lines.append("=" * 60)
        
        total = len(self.approval_history)
        approved = sum(1 for a in self.approval_history if a.get('approved', False))
        rejected = total - approved
        
        lines.append(f"Total requests: {total}")
        lines.append(f"Approved: {approved}")
        lines.append(f"Rejected: {rejected}")
        lines.append("")
        
        lines.append("Recent approvals:")
        for entry in self.approval_history[-3:]:
            status = "APPROVED" if entry.get('approved', False) else "REJECTED"
            lines.append(f"  - {entry.get('node')}: {status}")
        
        lines.append("=" * 60)
        return "\n".join(lines)

# ============================================================================
# BUILD HITL GRAPH
# ============================================================================

def build_hitl_graph():
    """
    Build a graph with Human-in-the-Loop capabilities.
    
    Returns:
        Compiled graph with HITL
    """
    print("\nBuilding HITL graph...")
    
    # Create HITL manager
    hitl = HumanInTheLoop()
    
    # Create graph
    graph = StateGraph(OrchestrationState)
    
    # Add nodes with HITL wrappers
    print("  Adding nodes with HITL...")
    graph = hitl.add_approval_node(graph, "supervisor", supervisor_node)
    graph = hitl.add_approval_node(graph, "research", research_node)
    graph = hitl.add_approval_node(graph, "analysis", analysis_node)
    graph = hitl.add_approval_node(graph, "writing", writing_node)
    
    # Add error and finalize nodes
    graph.add_node("error", error_node)
    graph.add_node("finalize", finalize_node)
    
    # Add edges
    print("  Adding edges...")
    graph.add_conditional_edges(
        "supervisor",
        route_after_supervisor,
        {
            "research": "research",
            "analysis": "analysis",
            "writing": "writing",
            "finalize": "finalize",
            "error": "error"
        }
    )
    
    graph.add_conditional_edges(
        "research",
        route_after_agent,
        {
            "analysis": "analysis",
            "writing": "writing",
            "finalize": "finalize",
            "error": "error"
        }
    )
    
    graph.add_conditional_edges(
        "analysis",
        route_after_agent,
        {
            "writing": "writing",
            "finalize": "finalize",
            "error": "error"
        }
    )
    
    graph.add_conditional_edges(
        "writing",
        route_after_agent,
        {
            "finalize": "finalize",
            "error": "error"
        }
    )
    
    graph.add_edge("error", END)
    graph.add_edge("finalize", END)
    
    # Set entry point
    graph.set_entry_point("supervisor")
    
    # Compile with checkpointer if available
    if MemorySaver:
        memory = MemorySaver()
        compiled_graph = graph.compile(checkpointer=memory)
    else:
        memory = SimpleMemorySaver()
        compiled_graph = graph.compile(checkpointer=memory)
    
    print("  HITL graph compiled successfully.")
    return compiled_graph, hitl

# Build HITL graph
print("\nBuilding HITL graph...")
hitl_graph, hitl_manager = build_hitl_graph()

print("\nHITL Graph Structure:")
print("=" * 40)
print("  START")
print("    |")
print("    v")
print("  SUPERVISOR (HITL)")
print("    |")
print("    +---> RESEARCH (HITL)")
print("    |       |")
print("    |       +---> ANALYSIS (HITL)")
print("    |       |       |")
print("    |       |       +---> WRITING (HITL)")
print("    |       |               |")
print("    |       |               v")
print("    +------>--------------> FINALIZE")
print("                                |")
print("                                v")
print("                              ERROR")
print("                                |")
print("                                v")
print("                               END")
print("=" * 40)

print("\n" + "=" * 60)
print("Human-in-the-Loop implementation complete.")
print("HITL manager created with approval tracking.")

Implementing Human-in-the-Loop...

Building HITL graph...

Building HITL graph...
  Adding nodes with HITL...
  Adding edges...
  HITL graph compiled successfully.

HITL Graph Structure:
  START
    |
    v
  SUPERVISOR (HITL)
    |
    +---> RESEARCH (HITL)
    |       |
    |       +---> ANALYSIS (HITL)
    |       |       |
    |       |       +---> WRITING (HITL)
    |       |               |
    |       |               v
    +------>--------------> FINALIZE
                                |
                                v
                              ERROR
                                |
                                v
                               END

Human-in-the-Loop implementation complete.
HITL manager created with approval tracking.


In [12]:
# CELL 9: Complete Fixed Orchestration System
# This cell provides a complete working orchestration system

import gradio as gr
from datetime import datetime
import time
import re
import math
import requests
import wikipediaapi
from typing import Dict, Any, Optional, List, Tuple, TypedDict
from langgraph.graph import StateGraph, END

print("BUILDING COMPLETE FIXED ORCHESTRATION SYSTEM")
print("=" * 60)

# ============================================================================
# TOOLS
# ============================================================================

class CalculatorTool:
    name = "calculator"
    def _run(self, expression: str) -> str:
        try:
            safe_context = {'__builtins__': {}, 'math': math, 'sqrt': math.sqrt, 
                          'sin': math.sin, 'cos': math.cos, 'tan': math.tan,
                          'log': math.log, 'log10': math.log10, 'abs': abs,
                          'pi': math.pi, 'e': math.e}
            result = eval(expression, safe_context)
            return f"Result: {float(result)}"
        except Exception as e:
            return f"Error: {str(e)}"

class WikipediaTool:
    name = "wikipedia_search"
    def __init__(self):
        self._wiki = wikipediaapi.Wikipedia(language='en', user_agent='Orchestration/1.0')
    def _run(self, query: str, max_results: int = 2) -> str:
        try:
            page = self._wiki.page(query)
            if page.exists():
                return f"Wikipedia: {page.title}\n{page.summary[:300]}..."
            results = list(self._wiki.search(query))[:max_results]
            if not results:
                return f"No results for: {query}"
            output = f"Results for '{query}':\n"
            for title in results:
                page = self._wiki.page(title)
                if page.exists():
                    output += f"- {title}: {page.summary[:150]}...\n"
            return output
        except Exception as e:
            return f"Error: {str(e)}"

class WebSearchTool:
    name = "web_search"
    def _run(self, query: str, max_results: int = 2) -> str:
        try:
            url = "https://en.wikipedia.org/w/api.php"
            params = {"action": "query", "list": "search", "srsearch": query, 
                     "format": "json", "srlimit": max_results}
            response = requests.get(url, params=params, timeout=10)
            if response.status_code != 200:
                return f"No results for: {query}"
            data = response.json()
            results = data.get("query", {}).get("search", [])
            if not results:
                return f"No results for: {query}"
            output = f"Search Results for '{query}':\n"
            for item in results:
                title = item.get("title", "")
                snippet = re.sub(r'<[^>]+>', '', item.get("snippet", ""))
                output += f"- {title}: {snippet[:150]}...\n"
            return output
        except Exception as e:
            return f"Error: {str(e)}"

class ToolRegistry:
    def __init__(self):
        self._tools = {}
    def register_tool(self, tool):
        self._tools[tool.name] = tool
    def get_tool(self, name):
        return self._tools.get(name)
    def execute_tool(self, name, **kwargs):
        tool = self.get_tool(name)
        if not tool:
            return {"success": False, "error": f"Tool {name} not found"}
        try:
            result = tool._run(**kwargs)
            return {"success": True, "result": result}
        except Exception as e:
            return {"success": False, "error": str(e)}

# ============================================================================
# AGENTS
# ============================================================================

class SimpleAgent:
    def __init__(self, name, role, registry):
        self.name = name
        self.role = role
        self.registry = registry
    
    def execute(self, input_data):
        if isinstance(input_data, str):
            query = input_data
        elif isinstance(input_data, dict):
            query = input_data.get('query', input_data.get('topic', str(input_data)))
        else:
            query = str(input_data)
        
        # Try Wikipedia
        result = self.registry.execute_tool("wikipedia_search", query=query)
        if result.get('success') and result.get('result'):
            return {"topic": query, "result": result['result'], "source": "Wikipedia"}
        
        # Try Web Search
        result = self.registry.execute_tool("web_search", query=query)
        if result.get('success') and result.get('result'):
            return {"topic": query, "result": result['result'], "source": "Web Search"}
        
        return {"topic": query, "result": f"Information about {query} not found.", "source": "System"}

# ============================================================================
# STATE
# ============================================================================

class AgentState(TypedDict):
    query: str
    current_node: str
    research_result: Optional[str]
    analysis_result: Optional[str]
    writing_result: Optional[str]
    final_response: Optional[str]
    status: str
    errors: List[str]

def create_state(query: str) -> AgentState:
    return {
        "query": query,
        "current_node": "start",
        "research_result": None,
        "analysis_result": None,
        "writing_result": None,
        "final_response": None,
        "status": "initialized",
        "errors": []
    }

# ============================================================================
# NODES
# ============================================================================

def research_node(state: AgentState) -> AgentState:
    print(f"\n[Research] Processing: {state['query']}")
    try:
        agent = SimpleAgent("ResearchAgent", "Researcher", registry)
        result = agent.execute(state['query'])
        state['research_result'] = result.get('result', 'No research found')
        state['current_node'] = 'research'
        state['status'] = 'research_complete'
        print(f"[Research] Complete")
    except Exception as e:
        state['errors'].append(f"Research error: {str(e)}")
        state['status'] = 'error'
    return state

def analyze_node(state: AgentState) -> AgentState:
    print(f"\n[Analysis] Processing")
    try:
        research = state.get('research_result', '')
        if not research or "not found" in research.lower():
            state['analysis_result'] = "No research data available for analysis."
        else:
            # Simple analysis - extract key points
            sentences = research.split('.')
            key_points = [s.strip() for s in sentences[:3] if len(s.strip()) > 20]
            analysis = f"Analysis of research:\n"
            for i, point in enumerate(key_points, 1):
                analysis += f"{i}. {point}\n"
            state['analysis_result'] = analysis if key_points else "Analysis: Research found but no key points extracted."
        state['current_node'] = 'analysis'
        state['status'] = 'analysis_complete'
        print(f"[Analysis] Complete")
    except Exception as e:
        state['errors'].append(f"Analysis error: {str(e)}")
        state['status'] = 'error'
    return state

def write_node(state: AgentState) -> AgentState:
    print(f"\n[Writing] Processing")
    try:
        analysis = state.get('analysis_result', '')
        query = state.get('query', '')
        
        if not analysis or "No research" in analysis:
            state['writing_result'] = f"No analysis available to write report for: {query}"
        else:
            report = f"REPORT ON: {query}\n"
            report += "=" * 50 + "\n\n"
            report += analysis + "\n"
            report += "-" * 50 + "\n"
            report += f"Report generated on: {datetime.now().isoformat()}\n"
            report += "=" * 50
            state['writing_result'] = report
        state['current_node'] = 'writing'
        state['status'] = 'writing_complete'
        print(f"[Writing] Complete")
    except Exception as e:
        state['errors'].append(f"Writing error: {str(e)}")
        state['status'] = 'error'
    return state

def finalize_node(state: AgentState) -> AgentState:
    print(f"\n[Finalize] Processing")
    try:
        report = state.get('writing_result', '')
        if not report or "No analysis" in report:
            report = state.get('analysis_result', 'No results available')
        if not report or "No research" in report:
            report = state.get('research_result', 'No results found')
        
        final = f"QUERY: {state['query']}\n\n"
        final += "=" * 60 + "\n"
        final += "FINAL RESPONSE\n"
        final += "=" * 60 + "\n\n"
        final += report
        final += "\n\n" + "=" * 60
        state['final_response'] = final
        state['status'] = 'complete'
        print(f"[Finalize] Complete")
    except Exception as e:
        state['errors'].append(f"Finalize error: {str(e)}")
        state['status'] = 'error'
    return state

# ============================================================================
# ROUTING
# ============================================================================

def route_after_research(state: AgentState) -> str:
    if state.get('status') == 'error':
        return 'error'
    return 'analyze'

def route_after_analyze(state: AgentState) -> str:
    if state.get('status') == 'error':
        return 'error'
    return 'write'

def route_after_write(state: AgentState) -> str:
    if state.get('status') == 'error':
        return 'error'
    return 'finalize'

def route_after_finalize(state: AgentState) -> str:
    return 'end'

# ============================================================================
# BUILD GRAPH
# ============================================================================

def build_graph():
    print("\nBuilding orchestration graph...")
    
    graph = StateGraph(AgentState)
    
    # Add nodes
    graph.add_node("research", research_node)
    graph.add_node("analyze", analyze_node)
    graph.add_node("write", write_node)
    graph.add_node("finalize", finalize_node)
    
    # Add edges
    graph.set_entry_point("research")
    graph.add_conditional_edges("research", route_after_research, {
        "analyze": "analyze",
        "error": "finalize"
    })
    graph.add_conditional_edges("analyze", route_after_analyze, {
        "write": "write",
        "error": "finalize"
    })
    graph.add_conditional_edges("write", route_after_write, {
        "finalize": "finalize",
        "error": "finalize"
    })
    graph.add_edge("finalize", END)
    
    compiled = graph.compile()
    print("Graph compiled successfully")
    return compiled

# ============================================================================
# REGISTRY
# ============================================================================

print("\nSetting up tools...")
registry = ToolRegistry()
registry.register_tool(WikipediaTool())
registry.register_tool(WebSearchTool())
registry.register_tool(CalculatorTool())
print(f"Tools registered: {registry._tools.keys()}")

# ============================================================================
# BUILD GRAPH
# ============================================================================

graph = build_graph()

# ============================================================================
# INTERFACE
# ============================================================================

print("\nCreating interface...")

def run_orchestration(query: str) -> str:
    if not query or query.strip() == "":
        return "Please enter a valid query."
    
    try:
        state = create_state(query)
        final_state = graph.invoke(state)
        
        response = final_state.get('final_response', 'No response generated')
        if not response or response == 'No response generated':
            response = f"Query: {query}\n\nResults:\n"
            if final_state.get('research_result'):
                response += final_state['research_result'][:300]
            if final_state.get('analysis_result'):
                response += "\n\n" + final_state['analysis_result'][:300]
            if final_state.get('writing_result'):
                response += "\n\n" + final_state['writing_result']
        
        if final_state.get('errors'):
            response += f"\n\nErrors: {', '.join(final_state['errors'])}"
        
        return response
    except Exception as e:
        return f"Error: {str(e)}"

with gr.Blocks(title="Day 4: LangGraph Orchestration", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # Day 4: Agent Orchestration with LangGraph
    
    **Simple Orchestration Flow:**
    1. Research Node -> 2. Analyze Node -> 3. Write Node -> 4. Finalize Node
    
    **Try queries like:**
    - "artificial intelligence"
    - "machine learning"
    - "python programming"
    - "quantum computing"
    - "deep learning"
    """)
    
    chatbot = gr.Chatbot(height=400)
    
    with gr.Row():
        msg = gr.Textbox(placeholder="Enter your query...", scale=8)
        submit = gr.Button("Run", variant="primary", scale=1)
    
    clear = gr.Button("Clear")
    
    def respond(message, history):
        if not message or message.strip() == "":
            return history, ""
        response = run_orchestration(message)
        history.append((message, response))
        return history, ""
    
    submit.click(respond, [msg, chatbot], [chatbot, msg])
    msg.submit(respond, [msg, chatbot], [chatbot, msg])
    clear.click(lambda: ([], ""), None, [chatbot, msg])

print("\n" + "=" * 60)
print("Launching Interface...")
print("=" * 60)

demo.launch(share=True)

print("\n" + "=" * 60)
print("DAY 4 COMPLETE!")
print("=" * 60)

BUILDING COMPLETE FIXED ORCHESTRATION SYSTEM

Setting up tools...
Tools registered: dict_keys(['wikipedia_search', 'web_search', 'calculator'])

Building orchestration graph...
Graph compiled successfully

Creating interface...

Launching Interface...
* Running on local URL:  http://127.0.0.1:7862
* Running on public URL: https://30a0341317c9e48855.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



DAY 4 COMPLETE!

[Research] Processing: deep learning
[Research] Complete

[Analysis] Processing
[Analysis] Complete

[Writing] Processing
[Writing] Complete

[Finalize] Processing
[Finalize] Complete

[Research] Processing: ilyas qadri
[Research] Complete

[Analysis] Processing
[Analysis] Complete

[Writing] Processing
[Writing] Complete

[Finalize] Processing
[Finalize] Complete

[Research] Processing: python 
[Research] Complete

[Analysis] Processing
[Analysis] Complete

[Writing] Processing
[Writing] Complete

[Finalize] Processing
[Finalize] Complete


In [13]:
# CELL 10: Fix Wikipedia Search and Improve System
# This cell fixes the Wikipedia search and adds better data handling

import re
import requests
import urllib.parse

print("FIXING WIKIPEDIA SEARCH AND IMPROVING SYSTEM")
print("=" * 60)

# ============================================================================
# FIXED WIKIPEDIA SEARCH
# ============================================================================

def fixed_wikipedia_search(query: str, max_results: int = 3) -> str:
    """
    Fixed Wikipedia search using direct API calls.
    """
    try:
        # Clean query
        query = query.strip()
        if not query:
            return "Please provide a search query."
        
        # Use Wikipedia API
        url = "https://en.wikipedia.org/w/api.php"
        params = {
            "action": "query",
            "list": "search",
            "srsearch": query,
            "format": "json",
            "srlimit": max_results,
            "srprop": "snippet|titlesnippet",
            "utf8": 1
        }
        
        headers = {
            "User-Agent": "Orchestration-System/1.0 (Educational Purpose)"
        }
        
        response = requests.get(url, params=params, headers=headers, timeout=10)
        
        if response.status_code != 200:
            return f"Wikipedia API error: Status {response.status_code}"
        
        data = response.json()
        results = data.get("query", {}).get("search", [])
        
        if not results:
            return f"No Wikipedia results found for: {query}"
        
        output = []
        for item in results[:max_results]:
            title = item.get("title", "Unknown")
            snippet = item.get("snippet", "")
            # Remove HTML tags
            snippet = re.sub(r'<[^>]+>', '', snippet)
            # Get page URL
            page_url = f"https://en.wikipedia.org/wiki/{urllib.parse.quote(title.replace(' ', '_'))}"
            
            output.append(f"Title: {title}")
            output.append(f"Summary: {snippet[:300]}...")
            output.append(f"URL: {page_url}")
            output.append("")
        
        return "\n".join(output)
        
    except requests.exceptions.Timeout:
        return f"Wikipedia API timeout for: {query}"
    except Exception as e:
        return f"Error searching Wikipedia: {str(e)}"

# ============================================================================
# FIXED RESEARCH AGENT
# ============================================================================

class FixedResearchAgent:
    """
    Fixed Research Agent with better Wikipedia search.
    """
    
    def __init__(self, registry):
        self.registry = registry
        self.name = "ResearchAgent"
    
    def execute(self, query: str) -> Dict[str, Any]:
        print(f"\n[Research] Searching: {query}")
        
        result = {
            "topic": query,
            "content": "",
            "source": "None",
            "success": False
        }
        
        # Try fixed Wikipedia search first
        print("  Trying Wikipedia...")
        wiki_result = fixed_wikipedia_search(query, max_results=2)
        
        if wiki_result and "No Wikipedia results" not in wiki_result and "Error" not in wiki_result:
            result["content"] = wiki_result
            result["source"] = "Wikipedia"
            result["success"] = True
            print("  Found Wikipedia results")
            return result
        
        # Try web search as fallback
        print("  Trying Web Search...")
        web_result = self.registry.execute_tool("web_search", query=query)
        
        if web_result.get('success') and web_result.get('result'):
            if "No results" not in web_result['result']:
                result["content"] = web_result['result']
                result["source"] = "Web Search"
                result["success"] = True
                print("  Found Web results")
                return result
        
        # If nothing found, provide helpful response
        result["content"] = f"""
No specific information found for '{query}'.

Suggestions:
1. Try a simpler query (e.g., 'AI' instead of 'artificial intelligence and its applications')
2. Check the spelling of your query
3. Try a different topic

Common topics that work well:
- Artificial Intelligence
- Machine Learning
- Python (programming language)
- Quantum Computing
- Deep Learning
- Neural Networks
- Data Science
"""
        result["source"] = "System"
        result["success"] = False
        print("  No results found, using system response")
        
        return result

# ============================================================================
# FIXED ANALYSIS AGENT
# ============================================================================

class FixedAnalysisAgent:
    """
    Fixed Analysis Agent with better information extraction.
    """
    
    def __init__(self, registry):
        self.registry = registry
        self.name = "AnalysisAgent"
    
    def execute(self, research_result: Dict[str, Any]) -> Dict[str, Any]:
        print(f"\n[Analysis] Analyzing: {research_result.get('topic', 'Unknown')}")
        
        content = research_result.get('content', '')
        topic = research_result.get('topic', 'Unknown')
        
        analysis = {
            "topic": topic,
            "content": "",
            "key_points": [],
            "summary": ""
        }
        
        if not content or not research_result.get('success', False):
            analysis["content"] = f"No research data available for analysis of '{topic}'."
            analysis["summary"] = f"Unable to analyze '{topic}' due to lack of research data."
            return analysis
        
        # Extract key points
        lines = content.split('\n')
        key_points = []
        
        for line in lines:
            line = line.strip()
            if line and len(line) > 10:
                # Look for bullet points or numbered items
                if line.startswith('•') or line.startswith('-') or line.startswith('*'):
                    clean = re.sub(r'^[•\-*\s]+', '', line)
                    if clean and len(clean) > 10:
                        key_points.append(clean)
                elif line.startswith('Title:') or line.startswith('Summary:'):
                    clean = re.sub(r'^(Title:|Summary:)\s*', '', line)
                    if clean and len(clean) > 10:
                        key_points.append(clean)
                elif len(line) > 30 and line.count(' ') > 3:
                    # Likely a sentence
                    if len(key_points) < 5:
                        key_points.append(line[:200])
        
        # Remove duplicates
        key_points = list(dict.fromkeys(key_points))
        
        # If no key points found, use first few sentences
        if not key_points:
            sentences = re.split(r'[.!?]', content)
            for sentence in sentences[:5]:
                sentence = sentence.strip()
                if 20 < len(sentence) < 200:
                    key_points.append(sentence)
        
        analysis["key_points"] = key_points[:5]
        
        # Generate summary
        if key_points:
            summary = f"Analysis of '{topic}' found {len(key_points)} key points:\n"
            for i, point in enumerate(key_points[:3], 1):
                summary += f"{i}. {point}\n"
            analysis["summary"] = summary
            analysis["content"] = summary
        else:
            analysis["content"] = f"Analysis of '{topic}' completed but no specific key points were identified."
            analysis["summary"] = f"Analysis of '{topic}' completed with limited findings."
        
        print(f"  Found {len(key_points)} key points")
        return analysis

# ============================================================================
# FIXED WRITING AGENT
# ============================================================================

class FixedWritingAgent:
    """
    Fixed Writing Agent with better report generation.
    """
    
    def __init__(self, registry):
        self.registry = registry
        self.name = "WritingAgent"
    
    def execute(self, analysis_result: Dict[str, Any]) -> Dict[str, Any]:
        print(f"\n[Writing] Writing report for: {analysis_result.get('topic', 'Unknown')}")
        
        topic = analysis_result.get('topic', 'Unknown')
        content = analysis_result.get('content', '')
        key_points = analysis_result.get('key_points', [])
        
        report = {
            "topic": topic,
            "content": "",
            "key_points": key_points,
            "full_report": ""
        }
        
        # Generate report
        lines = []
        lines.append(f"REPORT ON: {topic}")
        lines.append("=" * 60)
        lines.append("")
        lines.append(f"Generated: {datetime.now().isoformat()}")
        lines.append("")
        
        if key_points:
            lines.append("KEY FINDINGS:")
            lines.append("-" * 40)
            for i, point in enumerate(key_points, 1):
                lines.append(f"{i}. {point}")
            lines.append("")
        
        if content:
            lines.append("DETAILED ANALYSIS:")
            lines.append("-" * 40)
            lines.append(content)
            lines.append("")
        
        if not key_points and not content:
            lines.append(f"No specific information found for '{topic}'.")
            lines.append("")
            lines.append("Suggestions for better results:")
            lines.append("1. Use more specific queries")
            lines.append("2. Try different topics")
            lines.append("3. Check spelling")
        
        lines.append("=" * 60)
        lines.append("END OF REPORT")
        
        full_report = "\n".join(lines)
        report["full_report"] = full_report
        report["content"] = full_report
        
        print(f"  Report generated, length: {len(full_report)} characters")
        return report

# ============================================================================
# UPDATE THE SYSTEM
# ============================================================================

print("\nUpdating agents with fixes...")

# Create fixed agents
fixed_research = FixedResearchAgent(registry)
fixed_analysis = FixedAnalysisAgent(registry)
fixed_writing = FixedWritingAgent(registry)

# Update the node functions to use fixed agents
def research_node_fixed(state: AgentState) -> AgentState:
    print(f"\n[Research Node] Processing: {state['query']}")
    try:
        result = fixed_research.execute(state['query'])
        state['research_result'] = result.get('content', 'No research found')
        state['current_node'] = 'research'
        state['status'] = 'research_complete'
        print(f"  Research complete, source: {result.get('source', 'Unknown')}")
    except Exception as e:
        state['errors'].append(f"Research error: {str(e)}")
        state['status'] = 'error'
    return state

def analyze_node_fixed(state: AgentState) -> AgentState:
    print(f"\n[Analysis Node] Processing...")
    try:
        research_data = {"topic": state['query'], "content": state.get('research_result', '')}
        result = fixed_analysis.execute(research_data)
        state['analysis_result'] = result.get('content', 'No analysis available')
        state['current_node'] = 'analysis'
        state['status'] = 'analysis_complete'
        print(f"  Analysis complete")
    except Exception as e:
        state['errors'].append(f"Analysis error: {str(e)}")
        state['status'] = 'error'
    return state

def write_node_fixed(state: AgentState) -> AgentState:
    print(f"\n[Writing Node] Processing...")
    try:
        analysis_data = {"topic": state['query'], "content": state.get('analysis_result', '')}
        result = fixed_writing.execute(analysis_data)
        state['writing_result'] = result.get('full_report', 'No report generated')
        state['current_node'] = 'writing'
        state['status'] = 'writing_complete'
        print(f"  Writing complete")
    except Exception as e:
        state['errors'].append(f"Writing error: {str(e)}")
        state['status'] = 'error'
    return state

# ============================================================================
# REBUILD GRAPH WITH FIXED NODES
# ============================================================================

def build_fixed_graph():
    print("\nBuilding fixed orchestration graph...")
    
    graph = StateGraph(AgentState)
    
    # Add nodes with fixed functions
    graph.add_node("research", research_node_fixed)
    graph.add_node("analyze", analyze_node_fixed)
    graph.add_node("write", write_node_fixed)
    graph.add_node("finalize", finalize_node)
    
    # Add edges
    graph.set_entry_point("research")
    graph.add_conditional_edges("research", route_after_research, {
        "analyze": "analyze",
        "error": "finalize"
    })
    graph.add_conditional_edges("analyze", route_after_analyze, {
        "write": "write",
        "error": "finalize"
    })
    graph.add_conditional_edges("write", route_after_write, {
        "finalize": "finalize",
        "error": "finalize"
    })
    graph.add_edge("finalize", END)
    
    compiled = graph.compile()
    print("Fixed graph compiled successfully")
    return compiled

# Build fixed graph
fixed_graph = build_fixed_graph()

# Update the interface to use fixed graph
def run_fixed_orchestration(query: str) -> str:
    if not query or query.strip() == "":
        return "Please enter a valid query."
    
    try:
        state = create_state(query)
        final_state = fixed_graph.invoke(state)
        
        response = final_state.get('final_response', 'No response generated')
        if not response or response == 'No response generated':
            response = f"Query: {query}\n\n"
            if final_state.get('writing_result'):
                response += final_state['writing_result']
            elif final_state.get('analysis_result'):
                response += final_state['analysis_result']
            elif final_state.get('research_result'):
                response += final_state['research_result']
        
        if final_state.get('errors'):
            response += f"\n\nErrors: {', '.join(final_state['errors'])}"
        
        return response
    except Exception as e:
        return f"Error: {str(e)}"

print("\n" + "=" * 60)
print("System updated with fixes:")
print("  - Fixed Wikipedia search with direct API")
print("  - Better error handling for missing results")
print("  - Improved analysis and key point extraction")
print("  - Cleaner report generation")
print("=" * 60)
print("\nTry queries again:")
print("  - 'ilyas qadri' (will show suggestions)")
print("  - 'artificial intelligence' (will show Wikipedia results)")
print("  - 'python programming' (will show results)")
print("  - 'quantum computing' (will show results)")

FIXING WIKIPEDIA SEARCH AND IMPROVING SYSTEM

Updating agents with fixes...

Building fixed orchestration graph...
Fixed graph compiled successfully

System updated with fixes:
  - Fixed Wikipedia search with direct API
  - Better error handling for missing results
  - Improved analysis and key point extraction
  - Cleaner report generation

Try queries again:
  - 'ilyas qadri' (will show suggestions)
  - 'artificial intelligence' (will show Wikipedia results)
  - 'python programming' (will show results)
  - 'quantum computing' (will show results)


In [14]:
# CELL 11: Final Working Interface for Day 4
# This cell creates the final working interface with all fixes applied

import gradio as gr
from datetime import datetime
import time

print("BUILDING FINAL WORKING INTERFACE")
print("=" * 60)

# ============================================================================
# ORCHESTRATION FUNCTION
# ============================================================================

def run_orchestration_final(query: str) -> str:
    """
    Run the orchestration with the fixed graph.
    """
    if not query or query.strip() == "":
        return "Please enter a valid query."
    
    print(f"\n[{datetime.now().strftime('%H:%M:%S')}] Processing: {query}")
    
    try:
        # Create initial state
        state = create_state(query)
        
        # Run the graph
        start_time = time.time()
        final_state = fixed_graph.invoke(state)
        execution_time = time.time() - start_time
        
        # Get the response
        response = final_state.get('final_response', '')
        
        # If no final response, build one
        if not response or response == 'No response generated':
            response = f"QUERY: {query}\n"
            response += "=" * 60 + "\n"
            response += "RESULTS\n"
            response += "=" * 60 + "\n\n"
            
            # Get writing result
            writing = final_state.get('writing_result', '')
            if writing and "No specific information" not in writing:
                response += writing
            else:
                # Get analysis result
                analysis = final_state.get('analysis_result', '')
                if analysis and "No research" not in analysis:
                    response += analysis
                else:
                    # Get research result
                    research = final_state.get('research_result', '')
                    if research:
                        response += research
                    else:
                        response += f"No information found for '{query}'."
        
        # Add execution info
        response += f"\n\nExecution Time: {execution_time:.2f}s"
        response += f"\nStatus: {final_state.get('status', 'unknown')}"
        
        # Add errors if any
        errors = final_state.get('errors', [])
        if errors:
            response += f"\n\nWarnings: {', '.join(errors)}"
        
        return response
        
    except Exception as e:
        error_msg = f"Error processing query: {str(e)}"
        print(f"  Error: {error_msg}")
        return error_msg

# ============================================================================
# GRADIO INTERFACE
# ============================================================================

with gr.Blocks(title="LangGraph Orchestration System", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # Agent Orchestration with LangGraph
    
    **How it works:**
    1. **Research Node** - Searches Wikipedia and web for information
    2. **Analysis Node** - Extracts key points and insights
    3. **Writing Node** - Generates a structured report
    4. **Finalize Node** - Formats the final response
    
    **Try these queries:**
    - `artificial intelligence`
    - `machine learning`
    - `python programming`
    - `quantum computing`
    - `deep learning`
    - `neural networks`
    - `data science`
    
    **Note:** Works best with specific topics. If no results found, you'll get helpful suggestions.
    """)
    
    with gr.Row():
        with gr.Column(scale=2):
            chatbot = gr.Chatbot(
                height=450,
                label="Conversation",
                bubble_full_width=False
            )
            
            with gr.Row():
                msg = gr.Textbox(
                    placeholder="Enter a topic to research (e.g., 'artificial intelligence')...",
                    label="Query",
                    container=False,
                    scale=8
                )
                submit_btn = gr.Button("Research", variant="primary", scale=1)
            
            with gr.Row():
                clear_btn = gr.Button("Clear Conversation", size="sm", scale=1)
                sample_btn = gr.Button("Show Examples", size="sm", scale=1)
        
        with gr.Column(scale=1):
            status_box = gr.Markdown("**Status:** Ready")
            info_box = gr.Markdown("""
            ### System Status
            
            **Nodes:**
            - Research (Wikipedia + Web Search)
            - Analysis (Key Point Extraction)
            - Writing (Report Generation)
            - Finalize (Response Formatting)
            
            **Features:**
            - Real data from Wikipedia
            - Web search fallback
            - Error handling
            - Performance tracking
            """)
    
    def respond(message, history):
        if not message or message.strip() == "":
            return history, "Please enter a query."
        
        response = run_orchestration_final(message)
        history.append((message, response))
        return history, "Completed"
    
    def show_examples():
        examples = """
        ### Example Queries to Try:
        
        1. **artificial intelligence** - Wikipedia article with summary
        2. **machine learning** - Research and analysis
        3. **python programming** - Programming language info
        4. **quantum computing** - Advanced computing topic
        5. **deep learning** - Neural networks explanation
        6. **neural networks** - AI architecture
        7. **data science** - Data analysis field
        8. **natural language processing** - Text processing AI
        
        **Note:** Use specific topics for best results. General queries may return suggestions.
        """
        return examples
    
    def clear_chat():
        return [], "Cleared"
    
    submit_btn.click(
        respond,
        inputs=[msg, chatbot],
        outputs=[chatbot, status_box]
    )
    
    msg.submit(
        respond,
        inputs=[msg, chatbot],
        outputs=[chatbot, status_box]
    )
    
    clear_btn.click(
        clear_chat,
        outputs=[chatbot, status_box]
    )
    
    sample_btn.click(
        show_examples,
        outputs=[status_box]
    )

print("\n" + "=" * 60)
print("Launching Final Working Interface...")
print("=" * 60)

demo.launch(share=True)

print("\n" + "=" * 60)
print("DAY 4 COMPLETE!")
print("=" * 60)
print("LangGraph Orchestration System:")
print("  - Research Node: Wikipedia + Web Search")
print("  - Analysis Node: Key Point Extraction")
print("  - Writing Node: Report Generation")
print("  - Finalize Node: Response Formatting")
print("")
print("Features:")
print("  - Real data from Wikipedia")
print("  - Web search fallback")
print("  - Error handling with suggestions")
print("  - Clean report formatting")
print("  - Performance tracking")
print("")
print("Try queries like:")
print("  - 'artificial intelligence'")
print("  - 'machine learning'")
print("  - 'python programming'")
print("  - 'quantum computing'")
print("  - 'deep learning'")
print("=" * 60)


BUILDING FINAL WORKING INTERFACE

Launching Final Working Interface...
* Running on local URL:  http://127.0.0.1:7863
* Running on public URL: https://ccc629016f083f816d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



DAY 4 COMPLETE!
LangGraph Orchestration System:
  - Research Node: Wikipedia + Web Search
  - Analysis Node: Key Point Extraction
  - Writing Node: Report Generation
  - Finalize Node: Response Formatting

Features:
  - Real data from Wikipedia
  - Web search fallback
  - Error handling with suggestions
  - Clean report formatting
  - Performance tracking

Try queries like:
  - 'artificial intelligence'
  - 'machine learning'
  - 'python programming'
  - 'quantum computing'
  - 'deep learning'

[07:00:34] Processing: quantum computing

[Research Node] Processing: quantum computing

[Research] Searching: quantum computing
  Trying Wikipedia...
  Found Wikipedia results
  Research complete, source: Wikipedia

[Analysis Node] Processing...

[Analysis] Analyzing: quantum computing
  Analysis complete

[Writing Node] Processing...

[Writing] Writing report for: quantum computing
  Report generated, length: 329 characters
  Writing complete

[Finalize] Processing
[Finalize] Complete
